# Training an SNN with snnTorch-style loops and ESN neurons

This notebook shows how to use the evolved neurons from this repository inside a snnTorch-style training loop.

We will:

1. Load and frame the event-based NMNIST dataset.
2. Build a time-unrolled SNN with evolved ESN neurons.
3. Train and evaluate a classifier end to end.

The training split is restricted to a small subset by default so the notebook stays quick while still following a real event-data workflow.

## 1. Imports and requirements

This notebook requires additional python packages. You can install the tutorial dependencies with:

```bash
pip install -e .[examples]
```

In [ ]:
from pathlib import Path

import snntorch as snn
import tonic
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

from esn.neurons.clr import N2D2, N3D1

SEED = 7
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")
print(f"Loaded {snn.__name__}")

## 2. Load a small NMNIST workflow

NMNIST is a compact event-based digit dataset. Each sample is an event stream from a moving ATIS sensor. Here we use `tonic` to denoise the events and bin them into a fixed number of frames.

The `DataLoader` returns tensors with shape `[batch, time, polarity, height, width]`. The model below will permute each batch into snnTorch's common time-first convention: `[time, batch, polarity, height, width]`.

The first run may download NMNIST. After that, `tonic.DiskCachedDataset` stores the framed tensors so later runs start much faster.

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATA_ROOT = PROJECT_ROOT / ".data"
CACHE_ROOT = PROJECT_ROOT / ".cache" / "snntorch_nmnist"

n_steps = 25
batch_size = 64
train_subset = 1024
test_subset = 256
num_workers = 0
use_disk_cache = True


def maybe_subset(dataset, n_samples):
    if n_samples is None:
        return dataset
    return Subset(dataset, range(min(n_samples, len(dataset))))


def make_nmnist_dataset(train: bool, n_samples=None):
    transform = tonic.transforms.Compose([
        tonic.transforms.Denoise(filter_time=10_000),
        tonic.transforms.ToFrame(
            sensor_size=tonic.datasets.NMNIST.sensor_size,
            n_time_bins=n_steps,
        ),
    ])

    dataset = tonic.datasets.NMNIST(
        save_to=str(DATA_ROOT),
        train=train,
        transform=transform,
    )
    dataset = maybe_subset(dataset, n_samples)

    if use_disk_cache:
        split = "train" if train else "test"
        subset_name = "full" if n_samples is None else str(n_samples)
        cache_path = CACHE_ROOT / f"{split}_{subset_name}_{n_steps}_steps"
        dataset = tonic.DiskCachedDataset(dataset, cache_path=str(cache_path))

    return dataset


def make_loader(dataset, train: bool):
    generator = torch.Generator().manual_seed(SEED) if train else None
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=train,
        generator=generator,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=train,
    )


train_dataset = make_nmnist_dataset(train=True, n_samples=train_subset)
test_dataset = make_nmnist_dataset(train=False, n_samples=test_subset)
train_loader = make_loader(train_dataset, train=True)
test_loader = make_loader(test_dataset, train=False)

frames, labels = next(iter(train_loader))
print(f"Train samples: {len(train_dataset):,}")
print(f"Test samples:  {len(test_dataset):,}")
print(f"Frames: shape={tuple(frames.shape)}, dtype={frames.dtype}")
print(f"Labels: shape={tuple(labels.shape)}, first={labels[:8].tolist()}")

## 3. Define a snnTorch-style SNN

The model below is a simple time-unrolled classifier:

- A linear layer maps input spikes into a hidden current.
- `N2D2(...)` turns that current into hidden spikes.
- A final linear layer maps hidden spikes to class logits.

snnTorch tutorials commonly make the time loop explicit. We follow that pattern here by carrying the neuron state across time steps. The evolved neuron initializes its state automatically when a sequence starts with `state=None`.

The logits are averaged across time before the loss is computed.

In [ ]:
class SnnTorchESNClassifier(nn.Module):
    def __init__(
        self,
        input_features: int,
        hidden_features: int = 128,
        n_classes: int = 10,
        binarize_frames: bool = True,
    ):
        super().__init__()
        self.binarize_frames = binarize_frames
        self.flatten = nn.Flatten(start_dim=1)
        self.fc1 = nn.Linear(input_features, hidden_features, bias=False)
        self.neuron = N2D2(threshold=1.0)
        self.fc2 = nn.Linear(hidden_features, n_classes, bias=False)

    def prepare_frames(self, frames: torch.Tensor):
        frames = frames.float()
        if self.binarize_frames:
            frames = (frames > 0).to(frames.dtype)
        return frames.permute(1, 0, 2, 3, 4).contiguous()

    def forward(self, frames: torch.Tensor):
        # frames: [batch, time, polarity, height, width]
        frames = self.prepare_frames(frames)
        state = None
        logits_over_time = []

        for frame_t in frames:
            current1 = self.fc1(self.flatten(frame_t))
            hidden_spikes, state = self.neuron(current1, state)
            logits_t = self.fc2(hidden_spikes)
            logits_over_time.append(logits_t)

        return torch.stack(logits_over_time).mean(dim=0)


input_features = int(torch.tensor(tonic.datasets.NMNIST.sensor_size).prod().item())
model = SnnTorchESNClassifier(input_features=input_features).to(device)
print(model)

## 4. Train and evaluate

This is ordinary PyTorch training. The snnTorch-style detail is that the model consumes framed event data as an explicit time sequence and carries neuron state across time.

In [ ]:
def run_epoch(model, loader, optimizer=None, max_batches=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    loss_fn = nn.CrossEntropyLoss()

    for batch_idx, (frames, labels) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        frames = frames.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad(set_to_none=True)

        logits = model(frames)
        loss = loss_fn(logits, labels)

        if training:
            loss.backward()
            optimizer.step()

        batch_size = labels.numel()
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


n_epochs = 3
learning_rate = 1e-3
max_train_batches = None
max_eval_batches = None

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(1, n_epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer, max_batches=max_train_batches)
    test_loss, test_acc = run_epoch(model, test_loader, max_batches=max_eval_batches)
    print(
        f"epoch {epoch:02d} | "
        f"train loss {train_loss:.3f}, acc {train_acc:.3f} | "
        f"test loss {test_loss:.3f}, acc {test_acc:.3f}"
    )

## 5. Inspect predictions

The trained network returns one logit vector per sample. Higher logits indicate the predicted class.

In [ ]:
model.eval()
with torch.no_grad():
    frames, labels = next(iter(test_loader))
    logits = model(frames.to(device)).cpu()
    predictions = logits.argmax(dim=1)

print("labels:      ", labels[:16].tolist())
print("predictions: ", predictions[:16].tolist())
print("frames shape:", tuple(frames.shape))
print("logits shape:", tuple(logits.shape))

## 6. Swap in another evolved neuron

All evolved CLR neurons expose the same interface, so swapping neuron classes is local. For example, try replacing the spiking neuron in `SnnTorchESNClassifier`:

```python
N2D2(threshold=1.0)
```

with:

```python
N3D1(threshold=1.0)
```

Then rerun the model and training cells. Different evolved neurons may prefer different input scales, thresholds, or learning rates.
Optimally, these parameters should be selected during hyperparameter optimization.